## vLLM Inferenz Server (Simulator)

llm-d-inference-simist ein leichtgewichtiger, konfigurierbarer Echtzeitsimulator, der das Verhalten von vLLM nachbildet, ohne dass GPUs oder rechenintensive Modelle benötigt werden. 

Er fungiert als vollständig OpenAI-kompatibler Server und ermöglicht Entwicklern das Testen von Clients, Schedulern und Infrastruktur anhand realistischer Anfrage-Antwort-Zyklen, Token-Streaming und Latenzmustern.

* [vLLM-Simulator](https://llm-d.ai/docs/architecture/Components/inference-sim)


In [ ]:
%%bash
source ~/work/env.py
cat <<EOF | tee ~/work/env.py
OPENAI_API_KEY="$OPENAI_API_KEY"
HF_TOKEN=""
AI_KUBECONFIG="$AI_KUBECONFIG"
AI_MODEL="Qwen/Qwen3.5-4B"
AI_NAME="qwen"
AI_IP="${AI_IP}"
EOF

In [ ]:
%%bash
source ~/work/env.py
kubectl create ns llm-d-sim || true
kubectl apply -n llm-d-sim -f - <<EOF 
apiVersion: apps/v1
kind: Deployment
metadata:
  name: inference-sim
spec:
  replicas: 1
  selector:
    matchLabels:
      app: inference-sim
  template:
    metadata:
      labels:
        app: inference-sim
    spec:
      containers:
      - name: sim
        image: ghcr.io/llm-d/llm-d-inference-sim:v0.7.1
        args:
          - "--model"
          - "Qwen/Qwen2.5-0.5B-Instruct"
          - "--port"
          - "8000"
        ports:
        - containerPort: 8000
---
apiVersion: v1
kind: Service
metadata:
  name: inference-sim
spec:
  selector:
    app: inference-sim
  ports:
  - port: 80
    targetPort: 8000
EOF


Bevor wir den einfachen Test starten können, muss der Inferenz Server bereit sein:

In [ ]:
%%bash
source ~/work/env.py
kubectl get -n llm-d-sim pods,services
# kubectl ${AI_KUBECONFIG} -n vllm describe deployment/${AI_NAME}-vllm-deployment-vllm
kubectl -n llm-d-sim logs deployment/inference-sim --tail=10 || true
#kubectl -n llm-d-sim logs deployment/inference-sim -f

### Testen

Vorher muss base_url gesetzt werden.

In [ ]:
%%bash
source ~/work/env.py
kubectl patch svc inference-sim -n llm-d-sim -p '{"spec":{"type":"NodePort"}}'
echo "AI_BASE_URL=\"http://localhost:$(kubectl get -n llm-d-sim  service inference-sim -o jsonpath='{.spec.ports[0].nodePort}')/v1\"" | tee -a ~/work/env.py

In [ ]:
%run ~/work/env.py
from openai import OpenAI

client = OpenAI(
    api_key=OPENAI_API_KEY,
    base_url=AI_BASE_URL,   # z.B. http://localhost:32441/v1
)

response = client.chat.completions.create(
    model="Qwen/Qwen2.5-0.5B-Instruct",
    messages=[
        {"role": "user", "content": "Wer war John F. Kennedy?"}
    ],
)

print(response.choices[0].message.content)

---
### Weitere Beispiele

Leider unterstützt der Simulator nur das Chat API

---
### Aufräumen

In [ ]:
%%bash
source ~/work/env.py
kubectl delete -n llm-d-sim deployment/inference-sim || true
kubectl delete -n llm-d-sim service/inference-sim || true
kubectl delete ns llm-d-sim || true
